# Method 2 — Bi-Encoder (Kaggle T4)

Phase 2 của `docs/method2_plan.md`: train 2 vòng, pre-compute index, hiệu chỉnh ngưỡng. Ngân sách ~4h GPU.

**Ba quy tắc sống còn trên Kaggle** (§8 `docs/method2_plan.md`):

1. Bật **Save & Run All (Commit)** cho job dài — session tương tác bị ngắt sau ~20 phút không tương tác, commit run chạy nền đủ 12h.
2. Checkpoint mỗi 500 step vào `/kaggle/working`, và **luôn** hỗ trợ `resume_from`.
3. Cache model HuggingFace thành Kaggle Dataset (`BAAI/bge-m3` ~2.3GB) thay vì tải lại mỗi session.


In [ ]:
# ===== Cell 1: env =====
!pip install -q 'sentence-transformers>=3.0' peft transformers accelerate \
                jsonschema rank_bm25 datasets

import torch

free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), f'{free/1024**3:.1f} / {total/1024**3:.1f} GB free')
# Kỳ vọng: Tesla T4, ~15.0 GB free. T4 KHÔNG có bf16 → mọi config dùng fp16.


In [ ]:
# ===== Cell 2: mount code + data =====
# Đẩy repo và data lên Kaggle Dataset (private) trước.
!cp -r /kaggle/input/toolcalling-vi-src/src /kaggle/working/
!cp -r /kaggle/input/toolcalling-vi-src/configs /kaggle/working/
!cp -r /kaggle/input/toolcalling-vi-data/data /kaggle/working/data
%cd /kaggle/working

import json, os, glob, sys
sys.path.insert(0, '/kaggle/working')

# Cache model HF thành Kaggle Dataset để không tải lại mỗi session.
os.environ.setdefault('HF_HOME', '/kaggle/input/hf-cache')

manifest = json.load(open('data/method2/manifest.json', encoding='utf-8'))
print('snapshot commit:', manifest.get('git_commit'))


## Pre-flight — kiểm tra split/leakage TRƯỚC khi chạy full training

Benchmark gốc chia split theo **sample** chứ không theo query nên cùng một
query nằm ở nhiều split. Ba điều kiện phải đồng thời bằng 0 theo normalized
query: `train ∩ val`, `train ∩ test`, và — quan trọng nhất — `val ∩ test`.

Vì sao `val ∩ test` mới là rủi ro nặng nhất: dù không train trên query đó,
việc chọn checkpoint/hyperparameter bằng val vẫn khiến metric test lạc quan
hơn thực tế. Decontamination diễn ra ở tầng dataset của Method 2; thư mục
`data/benchmark_vi` **giữ nguyên** để tái lập và để bốn method vẫn được đánh
giá trên đúng cùng một tập test.


In [ ]:
stats = json.load(open('data/method2/biencoder/pairs_stats.json', encoding='utf-8'))
decon = stats['decontamination']

print('unique query/split :', stats['unique_queries_per_split'])
print('positive pairs     :', stats['n_positive_pairs'])
print('negative samples   :', stats['n_negative_samples'])
print('query trùng split  :', decon['n_overlapping_queries'], decon['overlapping_queries'])
print('sample bị loại     :', decon['rows_dropped_total'], decon['rows_dropped_by_transition'])
print('overlap còn lại    :', stats['split_overlap_after'])

dirty = {k: v for k, v in stats['split_overlap_after'].items() if v}
assert not dirty, f'Còn leakage giữa các split: {dirty}'
assert not stats['unseen_tools_leaked_into_train_positives']
print('\nOK — train ∩ val = train ∩ test = val ∩ test = 0')


## Round 1 — train với hard negative round-0

`CachedMultipleNegativesRankingLoss` (GradCache) cho effective batch 256 với
mini_batch 8. Gradient accumulation **không** thay thế được: nó chỉ chia nhỏ
update chứ không làm tăng số in-batch negative.


In [ ]:
RUN = '/kaggle/working/artifacts/method2/biencoder/run01'
resume = sorted(glob.glob(f'{RUN}/checkpoint-*'))[-1] if glob.glob(f'{RUN}/checkpoint-*') else None
print('resume from:', resume)

!python -m src.models.biencoder.train train \
    --config configs/method2/biencoder.yaml \
    --output-dir {RUN} \
    --resume-from {resume}


## Round 2 — mine hard negatives rồi train lại **từ base**

Lấy tool sai nhưng xếp hạng cao (bỏ top-1 để tránh false negative). Round 2
train lại từ checkpoint gốc, không train tiếp từ round 1.


In [ ]:
!python -m src.models.biencoder.train mine \
    --config configs/method2/biencoder.yaml \
    --model {RUN}/final

RUN2 = '/kaggle/working/artifacts/method2/biencoder/run02'
!sed -i 's#biencoder/train.jsonl#biencoder/train_mined.jsonl#' configs/method2/biencoder.yaml
!python -m src.models.biencoder.train train \
    --config configs/method2/biencoder.yaml --output-dir {RUN2}


## Pre-compute index + hiệu chỉnh ngưỡng trên **val**


In [ ]:
!python -m src.models.biencoder.index \
    --config configs/method2/biencoder.yaml --model {RUN2}/final

# τ và τ_call CHỈ được hiệu chỉnh trên val, rồi freeze trước khi chạy test.
!python -m src.models.biencoder.evaluate calibrate \
    --config configs/method2/biencoder.yaml \
    --model {RUN2}/final \
    --pairs data/method2/biencoder/val.jsonl \
    --output {RUN2}/thresholds.json


## Gate để qua Phase 3

| Metric | Tập | Ngưỡng |
|---|---|---|
| Recall@1 | custom `val_seen` | ≥ 0.90 |
| Recall@1 | custom `val_unseen` | ≥ 0.75 |
| Recall@5 | custom `val_unseen` | ≥ 0.92 |
| Negative Recall @ τ | custom val negative | ≥ 0.80 |

Không đạt → thử theo thứ tự: (a) thêm param name vào document text,
(b) tăng hard negative lên 8, (c) đổi sang `AITeamVN/Vietnamese_Embedding`,
(d) full fine-tune thay LoRA.


In [ ]:
!python -m src.models.biencoder.evaluate evaluate \
    --config configs/method2/biencoder.yaml \
    --model {RUN2}/final \
    --pairs data/method2/biencoder/val.jsonl \
    --output results/method2/metrics/biencoder_val.json

report = json.load(open('results/method2/metrics/biencoder_val.json', encoding='utf-8'))
for slice_name, metrics in report['by_source_key'].items():
    print(slice_name, {k: v for k, v in metrics.items() if 'recall@' in k or k == 'mrr'})


## Run manifest — chốt lại toàn bộ mục audit

Train xong mà không audit được thì coi như chưa train. Cell này gom: commit
SHA (kèm cờ dirty), config YAML thực tế, fingerprint dataset + tool pool,
query counts và overlap theo split, số positive/negative pair, checkpoint,
best step + metric đã dùng để chọn, VRAM peak, thời lượng train, và
Recall@1/@5/@10 + MRR. Thiếu mục nào thì `audit_complete.missing` liệt kê ra.

`checkpoint_selection.available_metrics` cho biết tên metric thật của
`InformationRetrievalEvaluator` ở phiên bản đang chạy — khai vào
`train.metric_for_best_model` cho lần chạy sau.


In [ ]:
!python -m src.models.run_manifest \
    --run-dir {RUN2} \
    --config configs/method2/biencoder.yaml \
    --stage biencoder \
    --report retrieval=results/method2/metrics/biencoder_val.json

manifest = json.load(open(f'{RUN2}/run_manifest.json', encoding='utf-8'))
missing = manifest['audit_complete']['missing']
print('thiếu:', missing or 'không thiếu mục nào')
print('Recall/MRR:', manifest.get('retrieval_gate', {}).get('metrics'))
print('VRAM peak MB:', manifest['train']['peak_vram_mb'])
print('thời lượng (giờ):', manifest['train']['duration_hours'])
print('chọn checkpoint:', manifest['train']['checkpoint_selection'])
assert not missing, f'Chưa đủ artifact để audit: {missing}'


In [ ]:
# ===== Lưu artifact =====
# Kaggle chỉ giữ /kaggle/working (20GB). Nén để tải về hoặc làm Dataset mới.
!tar czf /kaggle/working/biencoder_run.tar.gz -C /kaggle/working/artifacts/method2 .
!du -h /kaggle/working/biencoder_run.tar.gz
